# CLIP Joint Fine-tuning & OOD Detection Framework

本 Notebook 演示了基于 CLIP 的联合微调和分布外（OOD）检测完整流程：

1. **环境设置** - 导入依赖和配置参数
2. **模型初始化** - 加载 CLIP 模型并设置训练模式
3. **数据加载** - 加载 ID/OOD 数据集和参考数据集
4. **联合微调** - 多数据集联合微调 CLIP
5. **OOD 检测器** - 实现多种 OOD 检测方法
6. **特征缓存** - 高效特征提取和缓存
7. **实验一** - 多方法对比评估
8. **实验二** - 超参数调优
9. **实验三** - 两阶段路由（防灾难性遗忘）
10. **结果可视化** - 结果分析与可视化

## 1. 环境设置与导入

In [1]:
# ==========================================
# 1.1 基础依赖导入
# ==========================================
import os
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from tqdm import tqdm

# sklearn 相关
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import roc_auc_score, roc_curve, auc, accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold

# 可视化
import matplotlib.pyplot as plt

# ==========================================
# 1.2 自定义模块导入
# ==========================================
# 数据加载
from utils_data import (
    get_xtail_trainloader, get_transforms, 
    Flickr8kDataset, MergedReferenceDataset
)

# 模型相关
from models.clip import get_clip_model
from models.utils import feature_distillation_loss, EMASmooth, cross_modal_distillation_loss

from classifiers.da_classifier_builder import LDAClassifierBuilder, RegularQDAClassifierBuilder, LowRankQDAClassifierBuilder
from classifiers.gaussian_statistics import GaussianStatistics

print("✓ 所有依赖导入成功")

✓ 所有依赖导入成功


## 2. 配置与超参数

In [2]:
# ==========================================
# 2.1 随机种子设置（保证可复现性）
# ==========================================
def fix_random_seed(seed=42):
    """设置固定随机种子"""
    print(f"Setting fixed seed: {seed}")
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
fix_random_seed(SEED)

# ==========================================
# 2.2 数据集配置
# ==========================================
# ID: 用于联合微调 (In-Distribution)
# OOD: 用于评估检测能力 (Out-of-Distribution)
ID_DATASETS = ["caltech101", "flowers", "oxford_pets", "stanford_cars", "food101"]
OOD_DATASETS = ["dtd", "eurosat", "mnist", "sun397"]

print(f"ID Datasets (Training): {ID_DATASETS}")
print(f"OOD Datasets (Evaluation): {OOD_DATASETS}")

# ==========================================
# 2.3 训练参数配置
# ==========================================
ROOT = "/home/raoxuan/projects/data/X-TAIL/"  # 请修改为实际路径
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 数据加载参数
NUM_SHOTS = 16
BATCH_SIZE = 32
MAX_NUM_PER_TEST_DATASET = 1000

# 微调参数
ITERATIONS = 1000
LR = 1e-4
WEIGHT_DECAY = 3e-5

# LoRA 参数
LORA_RANK = 4
LORA_TYPE = "lora_nsp"

# 蒸馏参数
REFERENCE_DATASET = "flickr8k"  # 可设为 None 关闭蒸馏
FD_WEIGHT = 1.0
CD_WEIGHT = 1.0

# ==========================================
# 2.4 Args 配置类
# ==========================================
class Args:
    def __init__(self):
        self.dataset_sequence = ID_DATASETS + OOD_DATASETS
        self.root = ROOT
        self.num_shots = NUM_SHOTS
        self.batch_size = BATCH_SIZE
        self.max_num_per_test_dataset = MAX_NUM_PER_TEST_DATASET
        self.seed = SEED
        self.device = DEVICE
        self.iterations = ITERATIONS
        self.lr = LR
        self.weight_decay = WEIGHT_DECAY
        self.lora_rank = LORA_RANK
        self.lora_type = LORA_TYPE
        self.nsp_eps = 0.05
        self.nsp_weight = 0.02
        self.weight_temp = 1.0
        self.weight_kind = "log1p"
        self.weight_p = 1.0
        self.reference_dataset = REFERENCE_DATASET
        self.reference_batch_size = 32
        self.num_workers = 4
        self.fd_weight = FD_WEIGHT
        self.cd_weight = CD_WEIGHT

args = Args()
print(f"\n✓ 配置完成，使用设备: {DEVICE}")

Setting fixed seed: 42
ID Datasets (Training): ['caltech101', 'flowers', 'oxford_pets', 'stanford_cars', 'food101']
OOD Datasets (Evaluation): ['dtd', 'eurosat', 'mnist', 'sun397']

✓ 配置完成，使用设备: cuda


## 3. 模型初始化

In [3]:
# ==========================================
# 3.1 加载 CLIP 模型
# ==========================================
# 微调模型 (LoRA模式)
model, processor = get_clip_model(args, train_mode='lora')
model.to(DEVICE)

# 预训练模型 (Frozen, 用于蒸馏和路由回退)
model_pretrain, _ = get_clip_model(args, train_mode="frozen")
model_pretrain.to(DEVICE)

print(f"✓ 模型加载完成: LoRA模型 + Frozen预训练模型")

# ==========================================
# 3.2 辅助函数定义
# ==========================================
def encode_text(model, text, processor, device):
    """编码文本为特征向量"""
    text_inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True)
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
    return model.get_text_features(**text_inputs)

def encode_image(model, img):
    """编码图像为特征向量"""
    return model.get_image_features(img)

def zeroshot_classifier(model, classnames, templates, processor, device):
    """构建零样本分类器权重"""
    zeroshot_weights = []
    with torch.no_grad():
        for classname in classnames:
            classname = classname.replace('_', ' ')
            texts = [template(classname) for template in templates]
            class_embeddings = encode_text(model, texts, processor, device)
            class_embeddings = class_embeddings / class_embeddings.norm(dim=-1, keepdim=True)
            class_embedding = class_embeddings.mean(dim=0)
            class_embedding /= class_embedding.norm()
            zeroshot_weights.append(class_embedding)
    return torch.stack(zeroshot_weights, dim=1).to(device)

def get_optimizer(params, lr, weight_decay, iterations):
    """获取优化器和学习率调度器"""
    optimizer = optim.AdamW(params, lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=iterations, eta_min=lr/3)
    return optimizer, scheduler

print("✓ 辅助函数定义完成")

✓ 模型加载完成: LoRA模型 + Frozen预训练模型
✓ 辅助函数定义完成


## 4. 数据加载

In [ ]:
# ==========================================
# 4.1 参考数据集加载（用于蒸馏）
# ==========================================
def load_reference_dataset(args, model_pretrain, processor, device):
    """
    加载并缓存参考数据集（Flickr8K）用于蒸馏
    Returns: DataLoader or None
    """
    if args.iterations == 0 or args.reference_dataset != "flickr8k":
        print("Skipping reference dataset loading.")
        return None
    
    try:
        # 加载 Flickr8k
        ref_dataset_obj = Flickr8kDataset(root="/data1/open_datasets/flickr8k/")
        raw_ref_loader = ref_dataset_obj.return_loader(
            batch_size=32, shuffle=False, num_workers=4
        )
        
        # 缓存 Teacher 特征
        cached_imgs, cached_txts = [], []
        cached_t_img_feats, cached_t_txt_feats = [], []
        
        with torch.no_grad():
            for imgs, txts in tqdm(raw_ref_loader, desc="Caching Reference Data"):
                imgs = imgs.to(device)
                t_img_feat = encode_image(model_pretrain, imgs)
                t_img_feat = t_img_feat / t_img_feat.norm(dim=-1, keepdim=True)
                t_txt_feat = encode_text(model_pretrain, txts, processor, device)
                t_txt_feat = t_txt_feat / t_txt_feat.norm(dim=-1, keepdim=True)
                
                cached_imgs.append(imgs.cpu())
                cached_txts.extend(txts)
                cached_t_img_feats.append(t_img_feat.cpu())
                cached_t_txt_feats.append(t_txt_feat.cpu())
        
        merged_ref_dataset = MergedReferenceDataset(
            torch.cat(cached_imgs), cached_txts, 
            torch.cat(cached_t_img_feats), torch.cat(cached_t_txt_feats)
        )
        reference_loader = DataLoader(
            merged_ref_dataset, batch_size=32, shuffle=True, 
            num_workers=4, pin_memory=True
        )
        
        print(f"✓ Reference dataset loaded: {len(merged_ref_dataset)} samples")
        return reference_loader
        
    except Exception as e:
        print(f"Warning: Failed to load reference dataset ({e}). Distillation disabled.")
        return None

reference_loader = load_reference_dataset(args, model_pretrain, processor, DEVICE)

加载了 8091 个Flickr8K样本


Caching Reference Data: 100%|██████████| 253/253 [00:17<00:00, 14.78it/s]


✓ Reference dataset loaded: 8091 samples


## 5. 联合微调

In [6]:
# ==========================================
# 5.1 联合微调函数定义
# ==========================================
def train_joint_model(model, model_pretrain, id_datasets_list, args, processor, ref_loader):
    """
    多数据集联合微调
    
    策略: 维护多个 DataLoader，训练循环中随机采样一个数据集进行训练。
    这解决了不同数据集 Label ID 冲突的问题。
    """
    print(f"\n=== Starting Joint Fine-tuning on {len(id_datasets_list)} datasets ===")
    
    # 初始化历史记录
    history = {
        'iteration': [], 'total_loss': [], 'ce_loss': [],
        'fd_loss': [], 'cd_loss': [], 'cosine_sim': []
    }
    
    # 准备所有数据集的 Loader 和 Class Names
    train_loaders = []
    dataset_class_names = []
    train_transform, _ = get_transforms(id_datasets_list[0])
    
    for d_name in id_datasets_list:
        tr_loader, _, _, c_names = get_xtail_trainloader(
            root=args.root, dataset_name=d_name, 
            transform_train=train_transform, transform_test=None,
            num_shots=args.num_shots, batch_size=args.batch_size
        )
        train_loaders.append(tr_loader)
        dataset_class_names.append(c_names)
    
    train_iters = [iter(l) for l in train_loaders]
    
    # 预计算 Zero-shot Classifier Weights
    print("Pre-computing classifier weights...")
    templates = [lambda x: f"a photo of a {x}."]
    dataset_classifiers = []
    for c_names in dataset_class_names:
        w = zeroshot_classifier(model, c_names, templates, processor, args.device)
        dataset_classifiers.append(w)

    # 优化器
    optimizer, scheduler = get_optimizer(
        model.vision_model.get_params(), args.lr, args.weight_decay, args.iterations
    )
    
    logit_scale = model.logit_scale.detach()
    ema_loss = EMASmooth(0.95)
    ref_iter = iter(ref_loader) if ref_loader else None

    # 训练循环
    model.train()
    for i in range(args.iterations):
        # 随机选择数据集
        ds_idx = np.random.randint(len(id_datasets_list))
        
        # 获取 Batch
        try:
            images, labels = next(train_iters[ds_idx])
        except StopIteration:
            train_iters[ds_idx] = iter(train_loaders[ds_idx])
            images, labels = next(train_iters[ds_idx])
            
        images = images.to(args.device, non_blocking=True)
        labels = labels.to(args.device, non_blocking=True)
        current_classifier = dataset_classifiers[ds_idx]

        # 前向传播
        img_feats = encode_image(model, images)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        
        logits = logit_scale.exp() * (img_feats @ current_classifier)
        ce_loss = F.cross_entropy(logits, labels, label_smoothing=0.1)
        loss = ce_loss
        
        # 蒸馏损失
        fd_loss_value = 0.0
        cd_loss_value = 0.0
        cosine_sim_value = 0.0
        
        if ref_iter is not None:
            try:
                r_imgs, _, t_img_f, t_txt_f = next(ref_iter)
            except StopIteration:
                ref_iter = iter(ref_loader)
                r_imgs, _, t_img_f, t_txt_f = next(ref_iter)
            
            r_imgs = r_imgs.to(args.device)
            t_img_f = t_img_f.to(args.device)
            t_txt_f = t_txt_f.to(args.device)
            
            s_img_f = encode_image(model, r_imgs)
            s_img_f = s_img_f / s_img_f.norm(dim=-1, keepdim=True)
            
            cos_sim = F.cosine_similarity(t_img_f, s_img_f, dim=-1)
            cosine_sim_value = cos_sim.mean().item()
            
            l_fd = feature_distillation_loss(t_img_f, s_img_f)
            l_cd = cross_modal_distillation_loss(logit_scale, s_img_f, t_txt_f, t_img_f, t_txt_f, 2.0)
            
            fd_loss_value = l_fd.item()
            cd_loss_value = l_cd.item()
            loss += args.fd_weight * l_fd + args.cd_weight * l_cd

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        ema_loss.update(loss.item())
        
        # 记录历史
        history['iteration'].append(i + 1)
        history['total_loss'].append(loss.item())
        history['ce_loss'].append(ce_loss.item())
        history['fd_loss'].append(fd_loss_value)
        history['cd_loss'].append(cd_loss_value)
        history['cosine_sim'].append(cosine_sim_value)
        
        if (i + 1) % 50 == 0:
            print(f"Iter {i+1}/{args.iterations} | DS: {id_datasets_list[ds_idx]} | "
                  f"Loss: {ema_loss.get():.4f} | FD: {fd_loss_value:.4f} | "
                  f"CD: {cd_loss_value:.4f} | CosSim: {cosine_sim_value:.4f}")

    # 汇总结果
    summary = {
        'final_total_loss': history['total_loss'][-1],
        'final_ce_loss': history['ce_loss'][-1],
        'final_fd_loss': history['fd_loss'][-1],
        'final_cd_loss': history['cd_loss'][-1],
        'final_cosine_sim': history['cosine_sim'][-1]
    }
    
    print("\n=== Training Summary ===")
    print(f"Total iterations: {args.iterations}")
    print(f"Final total loss: {summary['final_total_loss']:.4f}")
    
    return {'model': model, 'history': history, 'summary': summary}

print("✓ 联合微调函数定义完成")

✓ 联合微调函数定义完成


In [7]:
# ==========================================
# 5.2 执行联合微调
# ==========================================
if args.iterations > 0:
    training_result = train_joint_model(
        model, model_pretrain, ID_DATASETS, args, processor, reference_loader
    )
    model = training_result['model']
    training_history = training_result['history']
    training_summary = training_result['summary']
    print("\n✓ Joint Fine-tuning Completed")
else:
    print("Skipping fine-tuning (ITERATIONS = 0). Using pre-trained model.")
    model.eval()


=== Starting Joint Fine-tuning on 5 datasets ===
mean: (0.48145466, 0.4578275, 0.40821073)
std: (0.26862954, 0.26130258, 0.27577711)
caltech101
Reading split from /home/raoxuan/projects/data/X-TAIL/Caltech101/split_zhou_Caltech101.json
Creating a 16-shot dataset
flowers
Reading split from /home/raoxuan/projects/data/X-TAIL/Flowers/split_zhou_OxfordFlowers.json
Creating a 16-shot dataset
oxford_pets
Reading split from /home/raoxuan/projects/data/X-TAIL/Pets/split_zhou_OxfordPets.json
Creating a 16-shot dataset
stanford_cars
Reading split from /home/raoxuan/projects/data/X-TAIL/StanfordCars/split_zhou_StanfordCars.json
Creating a 16-shot dataset
food101
Reading split from /home/raoxuan/projects/data/X-TAIL/Food/split_zhou_Food101.json
Creating a 16-shot dataset
Pre-computing classifier weights...


`sdpa` attention does not support `output_attentions=True` or `head_mask`. Please set your attention to `eager` if you want any of these features.


Iter 50/1000 | DS: stanford_cars | Loss: 2.2861 | FD: 0.0042 | CD: 0.0136 | CosSim: 0.9958
Iter 100/1000 | DS: caltech101 | Loss: 2.0090 | FD: 0.0083 | CD: 0.0202 | CosSim: 0.9917
Iter 150/1000 | DS: stanford_cars | Loss: 2.1929 | FD: 0.0088 | CD: 0.0333 | CosSim: 0.9912
Iter 200/1000 | DS: stanford_cars | Loss: 1.9693 | FD: 0.0096 | CD: 0.0187 | CosSim: 0.9904
Iter 250/1000 | DS: oxford_pets | Loss: 1.8754 | FD: 0.0077 | CD: 0.0168 | CosSim: 0.9923
Iter 300/1000 | DS: caltech101 | Loss: 1.8755 | FD: 0.0074 | CD: 0.0123 | CosSim: 0.9926
Iter 350/1000 | DS: food101 | Loss: 1.8594 | FD: 0.0078 | CD: 0.0132 | CosSim: 0.9922
Iter 400/1000 | DS: flowers | Loss: 1.8479 | FD: 0.0073 | CD: 0.0149 | CosSim: 0.9927
Iter 450/1000 | DS: flowers | Loss: 1.8455 | FD: 0.0063 | CD: 0.0157 | CosSim: 0.9937
Iter 500/1000 | DS: caltech101 | Loss: 1.6866 | FD: 0.0068 | CD: 0.0126 | CosSim: 0.9932
Iter 550/1000 | DS: flowers | Loss: 1.7835 | FD: 0.0056 | CD: 0.0099 | CosSim: 0.9944
Iter 600/1000 | DS: food

## 6. OOD 检测器实现

实现多种 OOD 检测方法：
- **Mahalanobis**: 基于马氏距离的检测器
- **Classifier-based**: 基于自定义分类器模块的检测器

In [8]:
# ==========================================
# 6.1 Mahalanobis 距离 OOD 检测器
# ==========================================
class MahalanobisOODDetector:
    """
    Mahalanobis 距离 OOD 检测器
    
    设计思路：
    1. Fit: 提取所有 ID 训练数据的特征，计算每个 Class 的均值和共享协方差矩阵的逆
    2. Score: 给定测试图片特征 x，计算 min_c (x - mu_c)^T Sigma^{-1} (x - mu_c)
       距离越大，越可能是 OOD
    """
    
    def __init__(self, model):
        self.model = model
        self.class_means = None       # Shape: (C, D)
        self.class_precisions = None  # Shape: (C, D, D)
        self.fitted = False
    
    @torch.no_grad()
    def fit(self, dataset_names, args, alpha=0.2):
        """
        从数据集训练检测器
        Args:
            alpha: 类特定协方差的权重 (0.1 表示 10% class-specific, 90% shared)
        """
        print(f"\nFitting Mahalanobis Detector (Alpha={alpha})...")
        self.model.eval()
        
        all_feats, all_labels = [], []
        transform, _ = get_transforms(dataset_names[0])
        label_offset = 0
        
        for d_name in dataset_names:
            tr_loader, _, _, c_names = get_xtail_trainloader(
                root=args.root, dataset_name=d_name, 
                transform_train=transform, transform_test=None,
                num_shots=args.num_shots, batch_size=32
            )
            
            for imgs, lbls in tqdm(tr_loader, desc=f"Extracting {d_name}", leave=False):
                imgs = imgs.to(args.device)
                feats = encode_image(self.model, imgs)
                feats = feats / feats.norm(dim=-1, keepdim=True)
                all_feats.append(feats.cpu())
                all_labels.append(lbls.cpu() + label_offset)
            
            label_offset += len(c_names)
        
        all_feats = torch.cat(all_feats).numpy()
        all_labels = torch.cat(all_labels).numpy()
        self._fit_from_features_internal(all_feats, all_labels, args.device, alpha)
    
    def fit_from_features(self, all_features, all_labels, device, alpha=0.2):
        """从预计算的特征训练检测器"""
        print(f"\nFitting Mahalanobis from cached features (Alpha={alpha})...")
        all_feats = all_features if isinstance(all_features, np.ndarray) else all_features.numpy()
        all_labels = all_labels if isinstance(all_labels, np.ndarray) else all_labels.numpy()
        self._fit_from_features_internal(all_feats, all_labels, device, alpha)
    
    def _fit_from_features_internal(self, all_feats, all_labels, device, alpha):
        """内部特征拟合逻辑"""
        unique_classes = np.unique(all_labels)
        num_classes = len(unique_classes)
        feat_dim = all_feats.shape[1]
        
        print(f"Total ID samples: {len(all_labels)}, Classes: {num_classes}, Dim: {feat_dim}")
        
        # 计算均值和 Shared Covariance
        means = []
        centered_data = []
        class_indices = {}
        
        for c in unique_classes:
            idx = (all_labels == c)
            class_indices[c] = idx
            data_c = all_feats[idx]
            mu = np.mean(data_c, axis=0)
            means.append(mu)
            centered_data.append(data_c - mu)
        
        means = np.array(means)
        centered_data = np.concatenate(centered_data, axis=0)
        sigma_shared = np.cov(centered_data, rowvar=False)
        
        # 计算插值后的精度矩阵
        precisions = []
        reg_epsilon = 1e-6 * np.eye(feat_dim)
        
        for c in tqdm(unique_classes, desc="Calculating Covariances"):
            data_c = all_feats[class_indices[c]]
            sigma_c = np.cov(data_c, rowvar=False)
            sigma_interpolated = alpha * sigma_c + (1 - alpha) * sigma_shared
            
            try:
                prec = np.linalg.inv(sigma_interpolated + reg_epsilon)
            except np.linalg.LinAlgError:
                prec = np.linalg.pinv(sigma_interpolated)
            precisions.append(prec)
        
        self.class_means = torch.from_numpy(means).float().to(device)
        self.class_precisions = torch.from_numpy(np.array(precisions)).float().to(device)
        self.fitted = True
        print("✓ Mahalanobis Detector Fitted")

    @torch.no_grad()
    def predict_score(self, loader, device):
        """从 DataLoader 计算 OOD 分数"""
        if not self.fitted:
            raise ValueError("Detector not fitted yet!")
        
        self.model.eval()
        scores = []
        
        for imgs, _ in tqdm(loader, desc="Scoring", leave=False):
            imgs = imgs.to(device)
            feats = encode_image(self.model, imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            batch_scores = self._compute_mahalanobis_distance(feats)
            scores.extend(batch_scores.cpu().numpy())
        
        return np.array(scores)
    
    @torch.no_grad()
    def predict_score_from_features(self, features, device, batch_size=512):
        """从预计算特征计算 OOD 分数"""
        if not self.fitted:
            raise ValueError("Detector not fitted yet!")
        
        features = features.to(device)
        total_samples = features.shape[0]
        scores = []
        
        for i in tqdm(range(0, total_samples, batch_size), desc="Scoring (cached)"):
            batch_feats = features[i:i+batch_size]
            batch_scores = self._compute_mahalanobis_distance(batch_feats)
            scores.extend(batch_scores.cpu().numpy())
        
        return np.array(scores)
    
    def _compute_mahalanobis_distance(self, feats):
        """计算马氏距离的核心逻辑"""
        B, D = feats.shape
        C = self.class_means.shape[0]
        
        # 计算到每个类的距离
        diff = feats.unsqueeze(1) - self.class_means.unsqueeze(0)  # (B, C, D)
        temp = torch.einsum('bcd,cde->bce', diff, self.class_precisions)  # (B, C, D)
        dists = (temp * diff).sum(dim=-1)  # (B, C)
        min_dists, _ = dists.min(dim=1)  # 取最近类的距离
        
        return min_dists

print("✓ MahalanobisOODDetector 定义完成")

✓ MahalanobisOODDetector 定义完成


In [9]:
# ==========================================
# 6.2 LDA OOD 检测器 (sklearn)
# ==========================================
class LDA_OODDetector:
    """基于 sklearn LDA 的 OOD 检测器"""
    
    def __init__(self, model):
        self.model = model
        self.lda = None
        self.fitted = False
    
    @torch.no_grad()
    def fit(self, dataset_names, args):
        """从数据集训练检测器"""
        print(f"\nFitting LDA Detector...")
        self.model.eval()
        
        all_feats, all_labels = [], []
        transform, _ = get_transforms(dataset_names[0])
        label_offset = 0
        
        for d_name in dataset_names:
            tr_loader, _, _, c_names = get_xtail_trainloader(
                root=args.root, dataset_name=d_name,
                transform_train=transform, transform_test=None,
                num_shots=args.num_shots, batch_size=32
            )
            
            for imgs, lbls in tqdm(tr_loader, desc=f"Extracting {d_name}", leave=False):
                imgs = imgs.to(args.device)
                feats = encode_image(self.model, imgs)
                feats = feats / feats.norm(dim=-1, keepdim=True)
                all_feats.append(feats.cpu().numpy())
                all_labels.append(lbls.cpu().numpy() + label_offset)
            
            label_offset += len(c_names)
        
        all_feats = np.concatenate(all_feats, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
        self._fit_internal(all_feats, all_labels)
    
    def fit_from_features(self, all_features, all_labels):
        """从预计算特征训练检测器"""
        print(f"\nFitting LDA from cached features...")
        all_feats = all_features if isinstance(all_features, np.ndarray) else all_features.numpy()
        all_labels = all_labels if isinstance(all_labels, np.ndarray) else all_labels.numpy()
        self._fit_internal(all_feats, all_labels)
    
    def _fit_internal(self, all_feats, all_labels):
        """内部拟合逻辑"""
        print(f"Training LDA on {len(all_labels)} samples, {all_feats.shape[1]} dims...")
        self.lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
        self.lda.fit(all_feats, all_labels)
        self.fitted = True
        print("✓ LDA Detector Fitted")

    @torch.no_grad()
    def predict_score(self, loader, device):
        """从 DataLoader 计算 OOD 分数 (1 - Max Posterior Probability)"""
        if not self.fitted:
            raise ValueError("LDA not fitted yet!")
        
        self.model.eval()
        scores = []
        
        for imgs, _ in tqdm(loader, desc="Scoring LDA", leave=False):
            imgs = imgs.to(device)
            feats = encode_image(self.model, imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            feats_np = feats.cpu().numpy()
            
            probs = self.lda.predict_proba(feats_np)
            max_probs = np.max(probs, axis=1)
            batch_scores = 1.0 - max_probs
            scores.extend(batch_scores)
        
        return np.array(scores)

    def predict_score_from_features(self, features):
        """从预计算特征计算 OOD 分数"""
        if not self.fitted:
            raise ValueError("LDA not fitted yet!")
        
        if torch.is_tensor(features):
            features = features.cpu().numpy()
        
        probs = self.lda.predict_proba(features)
        max_probs = np.max(probs, axis=1)
        return 1.0 - max_probs

print("✓ LDA_OODDetector 定义完成")

✓ LDA_OODDetector 定义完成


In [10]:
# ==========================================
# 6.3 基于 Classifier 模块的 OOD 检测器
# ==========================================
class ClassifierBasedOODDetector:
    """
    基于 Classifier 模块的分类器 OOD 检测器
    支持: lda, lowrank_qda, qda
    """
    
    def __init__(self, model, classifier_type="lda", **classifier_kwargs):
        """
        Args:
            model: CLIP 模型
            classifier_type: 分类器类型 ("lda", "lowrank_qda", "qda")
            classifier_kwargs: 分类器参数
        """
        self.model = model
        self.classifier_type = classifier_type
        self.classifier_kwargs = classifier_kwargs
        self.classifier = None
        self.fitted = False
    
    @torch.no_grad()
    def fit(self, dataset_names, args):
        """从数据集训练检测器"""
        print(f"\nFitting {self.classifier_type.upper()} OOD Detector...")
        self.model.eval()
        
        all_feats, all_labels = [], []
        transform, _ = get_transforms(dataset_names[0])
        label_offset = 0
        
        for d_name in dataset_names:
            tr_loader, _, _, c_names = get_xtail_trainloader(
                root=args.root, dataset_name=d_name,
                transform_train=transform, transform_test=None,
                num_shots=args.num_shots, batch_size=32
            )
            
            for imgs, lbls in tqdm(tr_loader, desc=f"Extracting {d_name}", leave=False):
                imgs = imgs.to(args.device)
                feats = encode_image(self.model, imgs)
                feats = feats / feats.norm(dim=-1, keepdim=True)
                all_feats.append(feats.cpu())
                all_labels.append(lbls.cpu() + label_offset)
            
            label_offset += len(c_names)
        
        all_feats = torch.cat(all_feats).numpy()
        all_labels = torch.cat(all_labels).numpy()
        self.fit_from_features(all_feats, all_labels, args.device)
    
    @torch.no_grad()
    def fit_from_features(self, all_features, all_labels, device):
        """从预计算特征训练检测器"""
        print(f"\nFitting {self.classifier_type.upper()} from cached features...")
        
        all_feats = all_features if isinstance(all_features, np.ndarray) else all_features.numpy()
        all_labels = all_labels if isinstance(all_labels, np.ndarray) else all_labels.numpy()
        
        # 计算每个类的统计量
        unique_classes = np.unique(all_labels)
        stats_dict = {}
        
        for c in unique_classes:
            idx = (all_labels == c)
            data_c = all_feats[idx]
            mu = np.mean(data_c, axis=0)
            cov = np.cov(data_c, rowvar=False)
            
            mu_tensor = torch.from_numpy(mu).float()
            cov_tensor = torch.from_numpy(cov).float()
            stats_dict[int(c)] = GaussianStatistics(mu_tensor, cov_tensor)
        
        # 构建分类器
        self.classifier = self._build_classifier(stats_dict, device)
        self.fitted = True
        print(f"✓ {self.classifier_type.upper()} Detector Fitted")
    
    def _build_classifier(self, stats_dict, device):
        """根据类型构建分类器"""
        if self.classifier_type == "lda":
            builder = LDAClassifierBuilder(
                reg_alpha=self.classifier_kwargs.get('reg_alpha', 0.3),
                device=device
            )
        elif self.classifier_type == "lowrank_qda":
            builder = LowRankQDAClassifierBuilder(
                qda_reg_alpha1=self.classifier_kwargs.get('qda_reg_alpha1', 0.2),
                qda_reg_alpha2=self.classifier_kwargs.get('qda_reg_alpha2', 0.2),
                qda_reg_alpha3=self.classifier_kwargs.get('qda_reg_alpha3', 0.2),
                rank=self.classifier_kwargs.get('rank', 64),
                device=device
            )
        elif self.classifier_type == "qda":
            builder = RegularQDAClassifierBuilder(
                qda_reg_alpha1=self.classifier_kwargs.get('qda_reg_alpha1', 0.2),
                qda_reg_alpha2=self.classifier_kwargs.get('qda_reg_alpha2', 0.2),
                qda_reg_alpha3=self.classifier_kwargs.get('qda_reg_alpha3', 0.2),
                device=device
            )
        else:
            raise ValueError(f"Unknown classifier type: {self.classifier_type}")
        
        return builder.build(stats_dict)
    
    @torch.no_grad()
    def predict_score(self, loader, device):
        """从 DataLoader 计算 OOD 分数"""
        if not self.fitted:
            raise ValueError(f"{self.classifier_type} not fitted yet!")
        
        self.model.eval()
        scores = []
        
        for imgs, _ in tqdm(loader, desc=f"Scoring {self.classifier_type}", leave=False):
            imgs = imgs.to(device)
            feats = encode_image(self.model, imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            
            probs = self.classifier.predict_proba(feats)
            max_probs = torch.max(probs, dim=1)[0]
            batch_scores = 1.0 - max_probs
            scores.extend(batch_scores.cpu().numpy())
        
        return np.array(scores)

    @torch.no_grad()
    def predict_score_from_features(self, features, device, batch_size=512):
        """从预计算特征计算 OOD 分数"""
        if not self.fitted:
            raise ValueError(f"{self.classifier_type} not fitted yet!")
        
        features = features.to(device)
        total_samples = features.shape[0]
        scores = []
        
        for i in tqdm(range(0, total_samples, batch_size), 
                      desc=f"Scoring {self.classifier_type} (cached)"):
            batch_feats = features[i:i+batch_size]
            probs = self.classifier.predict_proba(batch_feats)
            max_probs = torch.max(probs, dim=1)[0]
            batch_scores = 1.0 - max_probs
            scores.extend(batch_scores.cpu().numpy())
        
        return np.array(scores)

print("✓ ClassifierBasedOODDetector 定义完成")

✓ ClassifierBasedOODDetector 定义完成


## 7. 特征提取与缓存

In [11]:
# ==========================================
# 7.1 特征提取和缓存函数
# ==========================================
@torch.no_grad()
def extract_and_cache_features(model, dataset_names, args, transform, device, 
                                batch_size=256, use_train=True):
    """
    提取并缓存数据集的特征
    
    Args:
        model: CLIP模型
        dataset_names: 数据集名称列表
        args: 配置参数
        transform: 数据变换
        device: 设备
        batch_size: 批次大小
        use_train: True使用16-shot训练集，False使用完整测试集
    
    Returns:
        dict: {dataset_name: features_tensor}
    """
    print(f"\nExtracting features for: {dataset_names}")
    model.eval()
    features_cache = {}
    
    for d_name in tqdm(dataset_names, desc="Extracting features"):
        if use_train:
            tr_loader, _, _, _ = get_xtail_trainloader(
                root=args.root, dataset_name=d_name,
                transform_train=transform, transform_test=None,
                num_shots=args.num_shots, batch_size=batch_size
            )
            loader = tr_loader
        else:
            _, te_loader, _, _ = get_xtail_trainloader(
                root=args.root, dataset_name=d_name,
                transform_train=None, transform_test=transform,
                num_shots=args.num_shots, batch_size=batch_size
            )
            loader = te_loader
        
        all_feats = []
        for imgs, _ in loader:
            imgs = imgs.to(device)
            feats = encode_image(model, imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu())
        
        features_cache[d_name] = torch.cat(all_feats)
        print(f"  {d_name}: {features_cache[d_name].shape[0]} samples, dim={features_cache[d_name].shape[1]}")
    
    return features_cache

@torch.no_grad()
def extract_id_training_features(model, dataset_names, args, transform, device):
    """
    提取 ID 训练数据的特征和标签（用于检测器训练）
    
    Returns:
        tuple: (all_features, all_labels, label_offsets)
    """
    print("\nExtracting ID training features and labels...")
    model.eval()
    
    all_feats, all_labels = [], []
    label_offsets = {}
    label_offset = 0
    
    for d_name in tqdm(dataset_names, desc="Extracting ID data"):
        tr_loader, _, _, c_names = get_xtail_trainloader(
            root=args.root, dataset_name=d_name,
            transform_train=transform, transform_test=None,
            num_shots=args.num_shots, batch_size=32
        )
        
        for imgs, lbls in tr_loader:
            imgs = imgs.to(device)
            feats = encode_image(model, imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu())
            all_labels.append(lbls.cpu() + label_offset)
        
        label_offsets[d_name] = label_offset
        label_offset += len(c_names)
    
    return torch.cat(all_feats).numpy(), torch.cat(all_labels).numpy(), label_offsets

print("✓ 特征提取函数定义完成")

✓ 特征提取函数定义完成


## 8. 评估函数

In [13]:
# ==========================================
# 8.1 OOD 评估指标计算
# ==========================================
def calculate_ood_metrics(id_scores, ood_scores):
    """
    计算 OOD 检测的完整评估指标
    
    Args:
        id_scores: ID 样本的 OOD 分数数组
        ood_scores: OOD 样本的 OOD 分数数组
    
    Returns:
        dict: 包含 AUROC、FPR@95TPR、Detection Error 等指标
    """
    y_true = np.concatenate([np.zeros(len(id_scores)), np.ones(len(ood_scores))])
    y_scores = np.concatenate([id_scores, ood_scores])
    
    # AUROC
    auroc = roc_auc_score(y_true, y_scores)
    
    # ROC 曲线
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    
    # FPR@95TPR
    fpr_at_95_tpr = 1.0
    if np.any(tpr >= 0.95):
        fpr_at_95_tpr = fpr[np.where(tpr >= 0.95)[0][0]]
    
    # Detection Error
    detection_error = np.min(0.5 * (1 - tpr) + 0.5 * fpr)
    
    return {
        "auroc": auroc,
        "fpr_at_95_tpr": fpr_at_95_tpr,
        "detection_error": detection_error,
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds
    }

# ==========================================
# 8.2 多方法对比评估函数
# ==========================================
def compare_ood_methods(id_train_feats, id_train_labels, args, model, device):
    """
    对比多种 OOD 检测方法的性能
    
    Returns:
        dict: 包含所有方法的评估结果
    """
    print("\n=== Training OOD Detectors ===")
    
    # 初始化检测器
    detectors = {
        "Mahalanobis": MahalanobisOODDetector(model),
        "LDA (Classifier)": ClassifierBasedOODDetector(model, classifier_type="lda", reg_alpha=0.1),
        "Low-rank QDA": ClassifierBasedOODDetector(
            model, classifier_type="lowrank_qda",
            qda_reg_alpha1=0.2, qda_reg_alpha2=2.0, qda_reg_alpha3=0.4, rank=64
        )
    }
    
    # 训练检测器
    detectors["Mahalanobis"].fit_from_features(id_train_feats, id_train_labels, device, alpha=0.2)
    detectors["LDA (Classifier)"].fit_from_features(id_train_feats, id_train_labels, device)
    detectors["Low-rank QDA"].fit_from_features(id_train_feats, id_train_labels, device)
    
    print("\n✓ All detectors trained")
    return detectors

print("✓ 评估函数定义完成")

✓ 评估函数定义完成


## 9. 实验一：多方法 OOD 检测对比

In [ ]:
# ==========================================
# 9.1 提取 ID 训练特征
# ==========================================
train_transform, test_transform = get_transforms(ID_DATASETS[0])

id_train_feats, id_train_labels, _ = extract_id_training_features(
    model, ID_DATASETS, args, test_transform, DEVICE
)

# ==========================================
# 9.2 训练所有检测器
# ==========================================
detectors = compare_ood_methods(id_train_feats, id_train_labels, args, model, DEVICE)

# ==========================================
# 9.3 提取评估特征
# ==========================================
print("\n=== Caching Evaluation Features ===")
id_eval_features_cache = extract_and_cache_features(
    model, ID_DATASETS, args, test_transform, DEVICE, batch_size=256, use_train=True
)
ood_features_cache = extract_and_cache_features(
    model, OOD_DATASETS, args, test_transform, DEVICE, batch_size=256, use_train=True
)

# ==========================================
# 9.4 计算 ID 分数
# ==========================================
print("\n=== Computing ID Scores ===")
results = {name: {"id_scores": [], "ood_scores": {}, "metrics": {}} for name in detectors}

for method_name, detector in detectors.items():
    id_scores_list = []
    for d_name in ID_DATASETS:
        s = detector.predict_score_from_features(
             id_eval_features_cache[d_name], DEVICE, batch_size=512
            )
        id_scores_list.append(s)
    results[method_name]["id_scores"] = np.concatenate(id_scores_list)

# ==========================================
# 9.5 计算 OOD 分数并评估
# ==========================================
print("\n=== Evaluating OOD Detection Performance ===")
print(f"{'Dataset':<15} | {'Maha AUROC':<12} | {'LDA(cls) AUROC':<15} | {'Low-rank QDA':<14}")
print("-" * 85)

for d_name in OOD_DATASETS:
    for method_name, detector in detectors.items():
        # 计算 OOD 分数
        ood_s = detector.predict_score_from_features(
                ood_features_cache[d_name], DEVICE, batch_size=512)
        
        results[method_name]["ood_scores"][d_name] = ood_s
        
        # 计算指标
        id_s = results[method_name]["id_scores"]
        metrics = calculate_ood_metrics(id_s, ood_s)
        results[method_name]["metrics"][d_name] = metrics
    
    # 打印结果
    print(f"{d_name:<15} | {results['Mahalanobis']['metrics'][d_name]['auroc']*100:>10.2f}% | "
          f"{results['LDA (Classifier)']['metrics'][d_name]['auroc']*100:>13.2f}% | "
          f"{results['Low-rank QDA']['metrics'][d_name]['auroc']*100:>12.2f}%")

# ==========================================
# 9.6 全局指标汇总
# ==========================================
print("-" * 85)
global_results = {}

for method_name in detectors:
    all_ood = np.concatenate(list(results[method_name]["ood_scores"].values()))
    id_s = results[method_name]["id_scores"]
    global_results[method_name] = calculate_ood_metrics(id_s, all_ood)

print(f"{'Global AUROC':<15} | {global_results['Mahalanobis']['auroc']*100:>10.2f}% | "
      f"{global_results['LDA (Classifier)']['auroc']*100:>13.2f}% | "
      f"{global_results['Low-rank QDA']['auroc']*100:>12.2f}%")

print("\n=== Performance Summary ===")
print(f"{'Method':<20} | {'AUROC':<10} | {'FPR@95%TPR':<12} | {'Det. Error':<10}")
print("-" * 70)
for method_name in detectors:
    r = global_results[method_name]
    print(f"{method_name:<20} | {r['auroc']*100:>8.2f}% | {r['fpr_at_95_tpr']*100:>10.2f}% | {r['detection_error']*100:>8.2f}%")

mean: (0.48145466, 0.4578275, 0.40821073)
std: (0.26862954, 0.26130258, 0.27577711)

Extracting ID training features and labels...


Extracting ID data:   0%|          | 0/5 [00:00<?, ?it/s]

caltech101
Reading split from /home/raoxuan/projects/data/X-TAIL/Caltech101/split_zhou_Caltech101.json
Creating a 16-shot dataset


Extracting ID data:  20%|██        | 1/5 [00:03<00:15,  3.75s/it]

flowers
Reading split from /home/raoxuan/projects/data/X-TAIL/Flowers/split_zhou_OxfordFlowers.json
Creating a 16-shot dataset


...

eurosat         |      99.25% |         99.13% |        99.73%


Scoring lowrank_qda (cached): 100%|██████████| 1/1 [00:00<00:00, 1384.26it/s]


mnist           |      98.85% |         99.90% |        99.96%


Scoring lowrank_qda (cached): 100%|██████████| 13/13 [00:00<00:00, 719.56it/s]


sun397          |      99.51% |         95.94% |        97.76%
-------------------------------------------------------------------------------------
Global AUROC    |      99.27% |         96.36% |        97.98%

=== Performance Summary ===
Method               | AUROC      | FPR@95%TPR   | Det. Error
----------------------------------------------------------------------
Mahalanobis          |    99.27% |       2.46% |     2.96%
LDA (Classifier)     |    96.36% |      24.49% |     8.25%
Low-rank QDA         |    97.98% |       9.62% |     5.76%


## 10. 实验二：超参数调优

使用 **In-Distribution Cross-Validation** 策略，不偷看 OOD 数据集。
目标是寻找使 ID 验证集异常分数尽可能低的参数。

In [16]:
# ==========================================
# 10.1 超参数调优器
# ==========================================
class EfficientHyperparameterTuner:
    """高效超参数调优器（使用 ID 交叉验证）"""
    
    def __init__(self, id_features, id_labels, model, device, n_folds=3):
        self.id_features = id_features
        self.id_labels = id_labels
        self.model = model
        self.device = device
        self.n_folds = n_folds
        self.best_params = {}

    def run_cv(self, detector_name, param_name, param_values, base_kwargs=None):
        """
        执行交叉验证
        
        Args:
            detector_name: 'Mahalanobis', 'LDA_cls', 'LowRank_QDA'
            param_name: 要搜索的参数名
            param_values: 参数值列表
            base_kwargs: 其他固定参数
        """
        if base_kwargs is None:
            base_kwargs = {}
        
        print(f"\n>>> Tuning {detector_name} parameter: {param_name}...")
        
        skf = StratifiedKFold(n_splits=self.n_folds, shuffle=True, random_state=SEED)
        results = {}
        
        for val in param_values:
            fold_scores = []
            current_kwargs = base_kwargs.copy()
            current_kwargs[param_name] = val
            
            for train_idx, val_idx in skf.split(self.id_features, self.id_labels):
                X_train, y_train = self.id_features[train_idx], self.id_labels[train_idx]
                X_val, y_val = self.id_features[val_idx], self.id_labels[val_idx]
                
                try:
                    score_metric = self._evaluate_detector(
                        detector_name, X_train, y_train, X_val, current_kwargs
                    )
                    fold_scores.append(score_metric)
                except Exception as e:
                    fold_scores.append(-np.inf)
            
            avg_score = np.mean(fold_scores)
            results[val] = avg_score
            print(f"  {param_name}={val}: ID Fit Score = {avg_score:.4f}")
        
        best_val = max(results, key=results.get)
        print(f"*** Best {param_name}: {best_val}")
        return best_val
    
    def _evaluate_detector(self, detector_name, X_train, y_train, X_val, kwargs):
        """评估单个检测器配置"""
        # 初始化检测器
        if detector_name == 'Mahalanobis':
            det = MahalanobisOODDetector(self.model)
            det.fit_from_features(X_train, y_train, self.device, **kwargs)
            val_scores = det.predict_score_from_features(
                torch.from_numpy(X_val).float(), self.device, batch_size=1024
            )
        
        
        elif detector_name == 'LDA_cls':
            det = ClassifierBasedOODDetector(self.model, classifier_type='lda', **kwargs)
            det.fit_from_features(X_train, y_train, self.device)
            val_scores = det.predict_score_from_features(
                torch.from_numpy(X_val).float(), self.device, batch_size=1024
            )
        
        elif detector_name == 'LowRank_QDA':
            det = ClassifierBasedOODDetector(self.model, classifier_type='lowrank_qda', **kwargs)
            det.fit_from_features(X_train, y_train, self.device)
            val_scores = det.predict_score_from_features(
                torch.from_numpy(X_val).float(), self.device, batch_size=1024
            )
        
        # ID 数据的 OOD score 越低越好，取负均值方便比较
        return -np.mean(val_scores)

print("✓ EfficientHyperparameterTuner 定义完成")

✓ EfficientHyperparameterTuner 定义完成


In [17]:
# ==========================================
# 10.2 执行超参数调优
# ==========================================
print("\n" + "="*80)
print(" Experiment 2: Hyperparameter Tuning ")
print("="*80)

tuner = EfficientHyperparameterTuner(id_train_feats, id_train_labels, model, DEVICE, n_folds=3)

# 1. Mahalanobis: 搜索 alpha
best_maha_alpha = tuner.run_cv('Mahalanobis', 'alpha', [0.0, 0.1, 0.2, 0.5])

# 3. Low-Rank QDA: 搜索 qda_reg_alpha2
best_lrqda_alpha2 = tuner.run_cv(
    'LowRank_QDA', 'qda_reg_alpha2', [0.1, 0.2, 0.5, 1.0, 5.0, 10.0],
    base_kwargs={'qda_reg_alpha1': 0.1, 'qda_reg_alpha3': 0.5, 'rank': 64}
)

# ==========================================
# 10.3 使用最佳参数构建最终检测器
# ==========================================
print("\n=== Building Final Detector with Best Parameters ===")

final_detector = ClassifierBasedOODDetector(
    model, 
    classifier_type='lowrank_qda',
    qda_reg_alpha1=0.1,
    qda_reg_alpha2=best_lrqda_alpha2,
    qda_reg_alpha3=0.5,
    rank=64
)
final_detector.fit_from_features(id_train_feats, id_train_labels, DEVICE)
print("✓ Final detector trained with optimized parameters")


 Experiment 2: Hyperparameter Tuning 

>>> Tuning Mahalanobis parameter: alpha...

Fitting Mahalanobis from cached features (Alpha=0.0)...
Total ID samples: 5717, Classes: 536, Dim: 512


Calculating Covariances: 100%|██████████| 536/536 [00:05<00:00, 95.37it/s]


✓ Mahalanobis Detector Fitted


Scoring (cached): 100%|██████████| 3/3 [00:00<00:00, 80.93it/s]



Fitting Mahalanobis from cached features (Alpha=0.0)...
Total ID samples: 5717, Classes: 536, Dim: 512


...


Fitting LOWRANK_QDA from cached features...


✓ LOWRANK_QDA Detector Fitted


Scoring lowrank_qda (cached): 100%|██████████| 3/3 [00:00<00:00, 362.63it/s]

  qda_reg_alpha2=10.0: ID Fit Score = -0.9974
*** Best qda_reg_alpha2: 0.1

=== Building Final Detector with Best Parameters ===

Fitting LOWRANK_QDA from cached features...


✓ LOWRANK_QDA Detector Fitted
✓ Final detector trained with optimized parameters


In [18]:
best_lrqda_alpha2

0.1

## 11. 实验三：两阶段路由（防灾难性遗忘）

**核心逻辑**：
1. **特征提取**: 使用微调模型提取特征
2. **OOD 检测**: 基于微调特征计算异常分数
3. **路由决策**: 
   - ID 样本 → 使用 Fine-tuned Features @ Fine-tuned Text Weights
   - OOD 样本 → 回退到 Frozen Features @ Frozen Text Weights

In [19]:
# ==========================================
# 11.1 零样本路由实现
# ==========================================
class ZeroShotRouter:
    """
    防灾难性遗忘的两阶段路由器
    在 ID 数据上使用微调模型，在 OOD 数据上回退到预训练模型
    """
    
    def __init__(self, ft_model, frozen_model, detector, device, processor):
        self.ft_model = ft_model
        self.frozen_model = frozen_model
        self.detector = detector
        self.device = device
        self.processor = processor
        self.threshold = 0.5  # 默认阈值，需通过 calibrate_threshold 校准

    def calibrate_threshold(self, id_features, recall_target=0.95):
        """
        基于 ID 训练特征校准阈值
        确保指定比例的 ID 样本被保留（不被错误路由到 Frozen）
        """
        print(f"Calibrating routing threshold (target ID recall: {recall_target*100}%)...")
        
        if isinstance(id_features, np.ndarray):
            id_features = torch.from_numpy(id_features).float()
        
        scores = self.detector.predict_score_from_features(
            id_features, self.device, batch_size=1024
        )
        
        # 阈值是分数的第 recall_target 百分位
        self.threshold = np.percentile(scores, recall_target * 100)
        print(f"✓ Threshold calibrated: {self.threshold:.4f}")

    def get_classifier_head(self, model, class_names):
        """为特定数据集构建分类头"""
        templates = [lambda x: f"a photo of a {x}."]
        with torch.no_grad():
            weights = zeroshot_classifier(model, class_names, templates, self.processor, self.device)
        return weights

    @torch.no_grad()
    def predict_and_route(self, images, class_names):
        """
        对一个 Batch 的图像进行路由分类
        
        Returns:
            final_probs: 最终的分类概率
            routing_mask: (B,) bool, True 表示被路由到了 Frozen 模型 (OOD)
        """
        B = images.shape[0]
        
        # Step 1: 使用微调模型提取特征
        ft_feats = encode_image(self.ft_model, images)
        ft_feats = ft_feats / ft_feats.norm(dim=-1, keepdim=True)
        
        # Step 2: OOD 检测
        if isinstance(self.detector, LDA_OODDetector):
            scores = self.detector.predict_score_from_features(ft_feats.cpu().numpy())
        else:
            scores = self.detector.predict_score_from_features(ft_feats, self.device, batch_size=B)
        
        # 生成掩码: True = OOD (需要回退), False = ID (保持微调)
        is_ood_mask = scores > self.threshold
        
        # Step 3: 准备分类头
        ft_head = self.get_classifier_head(self.ft_model, class_names)
        fr_head = self.get_classifier_head(self.frozen_model, class_names)
        
        final_logits = torch.zeros(B, len(class_names)).to(self.device)
        
        # Step 4: 路由执行
        # Case A: ID 样本 -> 使用 FT 特征 + FT 头
        id_indices = np.where(~is_ood_mask)[0]
        if len(id_indices) > 0:
            logit_scale = self.ft_model.logit_scale.exp()
            sub_feats = ft_feats[id_indices]
            final_logits[id_indices] = logit_scale * (sub_feats @ ft_head)
        
        # Case B: OOD 样本 -> 回退到 Frozen 模型
        ood_indices = np.where(is_ood_mask)[0]
        if len(ood_indices) > 0:
            sub_images = images[ood_indices]
            fr_feats = encode_image(self.frozen_model, sub_images)
            fr_feats = fr_feats / fr_feats.norm(dim=-1, keepdim=True)
            
            logit_scale_pre = self.frozen_model.logit_scale.exp()
            final_logits[ood_indices] = logit_scale_pre * (fr_feats @ fr_head)
        
        probs = F.softmax(final_logits, dim=-1)
        return probs, is_ood_mask

print("✓ ZeroShotRouter 定义完成")

✓ ZeroShotRouter 定义完成


In [ ]:
# ==========================================
# 11.2 初始化 Router 并校准阈值
# ==========================================
router = ZeroShotRouter(model, model_pretrain, final_detector, DEVICE, processor)
router.calibrate_threshold(id_train_feats, recall_target=0.95)

# ==========================================
# 11.3 综合对比评估
# ==========================================
def evaluate_method_comparison(router, dataset_names, args):
    """
    对比三种方法的性能:
    1. FT-Only: 仅使用微调模型（灾难性遗忘基准）
    2. Frozen-Only: 仅使用冻结模型（Zero-shot 上限）
    3. Router: 我们的方法
    """
    print("\n" + "="*100)
    print(f"{'Dataset':<20} | {'Type':<6} | {'FT-Only':<10} | {'Frozen':<10} | {'Router':<10} | {'Routed%':<10}")
    print("-" * 100)
    
    results = []
    _, test_transform = get_transforms(ID_DATASETS[0])

    for d_name in dataset_names:
        is_id = d_name in ID_DATASETS
        d_type = "ID" if is_id else "OOD"
        
        _, loader, _, class_names = get_xtail_trainloader(
            root=args.root, dataset_name=d_name, 
            transform_train=None, transform_test=test_transform, 
            num_shots=args.num_shots, batch_size=32
        )
        
        all_preds_ft, all_preds_fr, all_preds_router, all_labels = [], [], [], []
        routed_count, total_count = 0, 0
        
        # 预计算分类头
        ft_head = router.get_classifier_head(router.ft_model, class_names)
        fr_head = router.get_classifier_head(router.frozen_model, class_names)
        
        for images, labels in tqdm(loader, desc=f"Eval {d_name}", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            B = images.shape[0]
            
            # Router 预测
            router_probs, is_ood_mask = router.predict_and_route(images, class_names)
            preds_router = router_probs.argmax(dim=-1)
            
            # FT-Only 基线
            ft_feats = encode_image(router.ft_model, images)
            ft_feats = ft_feats / ft_feats.norm(dim=-1, keepdim=True)
            logits_ft = router.ft_model.logit_scale.exp() * (ft_feats @ ft_head)
            preds_ft = logits_ft.argmax(dim=-1)
            
            # Frozen-Only 基线
            fr_feats = encode_image(router.frozen_model, images)
            fr_feats = fr_feats / fr_feats.norm(dim=-1, keepdim=True)
            logits_fr = router.frozen_model.logit_scale.exp() * (fr_feats @ fr_head)
            preds_fr = logits_fr.argmax(dim=-1)
            
            all_preds_router.extend(preds_router.cpu().numpy())
            all_preds_ft.extend(preds_ft.cpu().numpy())
            all_preds_fr.extend(preds_fr.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            routed_count += is_ood_mask.sum().item()
            total_count += B
        
        # 计算准确率
        acc_ft = accuracy_score(all_labels, all_preds_ft)
        acc_fr = accuracy_score(all_labels, all_preds_fr)
        acc_router = accuracy_score(all_labels, all_preds_router)
        routing_rate = routed_count / total_count
        
        print(f"{d_name:<20} | {d_type:<6} | {acc_ft*100:>8.2f}% | {acc_fr*100:>8.2f}% | "
              f"{acc_router*100:>8.2f}% | {routing_rate*100:>8.2f}%")
        
        results.append({
            'dataset': d_name, 'type': d_type,
            'acc_ft': acc_ft, 'acc_fr': acc_fr, 'acc_router': acc_router,
            'routing_rate': routing_rate
        })
    
    print("-" * 100)
    return results

# 执行评估
all_datasets = ID_DATASETS + OOD_DATASETS
eval_results = evaluate_method_comparison(router, all_datasets, args)

## 12. 结果可视化

In [ ]:
# ==========================================
# 12.1 绘制对比结果
# ==========================================
def plot_results(results):
    """绘制 Router 效果对比图"""
    datasets = [r['dataset'] for r in results]
    acc_ft = [r['acc_ft'] for r in results]
    acc_fr = [r['acc_fr'] for r in results]
    acc_router = [r['acc_router'] for r in results]
    
    x = np.arange(len(datasets))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(15, 6))
    rects1 = ax.bar(x - width, acc_ft, width, label='FT Only (Forgetting)', color='#ff9999')
    rects2 = ax.bar(x, acc_router, width, label='Router (Ours)', color='#66b3ff')
    rects3 = ax.bar(x + width, acc_fr, width, label='Frozen (Oracle)', color='#99ff99')
    
    ax.set_ylabel('Accuracy')
    ax.set_title('Catastrophic Forgetting Mitigation: Router Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim([0, 1.1])
    
    # 标注 ID 和 OOD 区域
    id_indices = [i for i, r in enumerate(results) if r['type'] == 'ID']
    ood_indices = [i for i, r in enumerate(results) if r['type'] == 'OOD']
    
    if id_indices and ood_indices:
        boundary = (max(id_indices) + min(ood_indices)) / 2
        ax.axvline(x=boundary, color='gray', linestyle='--', alpha=0.5)
        ax.text(min(id_indices) + len(id_indices)/2 - 0.5, 1.05, 
                'In-Distribution (ID)', ha='center', fontsize=10, color='green')
        ax.text(min(ood_indices) + len(ood_indices)/2 - 0.5, 1.05, 
                'Out-of-Distribution (OOD)', ha='center', fontsize=10, color='red')
    
    plt.tight_layout()
    plt.show()

# 绘制结果
plot_results(eval_results)

print("\n✓ Visualization completed")

## 13. 总结

本 Notebook 完成了以下实验：

### 实验一：多方法 OOD 检测对比
- 实现了 Mahalanobis、LDA (sklearn)、LDA (Classifier)、Low-rank QDA 四种检测器
- 使用特征缓存机制提高评估效率

### 实验二：超参数调优
- 使用 In-Distribution Cross-Validation 策略
- 不偷看 OOD 数据集，保证评估的公平性

### 实验三：两阶段路由
- 实现 ZeroShotRouter 防灾难性遗忘
- ID 样本使用微调模型，OOD 样本回退到预训练模型
- 在保持 ID 性能的同时，恢复 OOD 上的泛化能力